# GO1 PACT-Pos → PACT hot-start checkpoint

Convert a trained GO1 PACT-Pos checkpoint into the weights-only checkpoint expected by the GO1 PACT `pretrained_path` loader. The conversion transfers the context encoder, actor trunk, critic, position head, learned torque head, privileged decoder, and separate GRF decoder.

The source position-action noise is retained. PACT's new torque-action noise is initialized with `INITIAL_TORQUE_STD`. Optimizer states are intentionally omitted because PACT and PACT-Pos do not have compatible optimizer layouts.

In [1]:
from pathlib import Path
import sys
import torch


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'rsl_rl').is_dir() and (candidate / 'legged_gym').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the HCR_Genesis_PACT_Development repository')


ROOT = find_repo_root()
MODULES_DIR = ROOT / 'rsl_rl' / 'modules'
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))

from actor_critic_pact import ActorCritic_PACT, ContextDecoder
from actor_critic_pact_pos import ActorCritic_PACT_Pos

torch.manual_seed(1)
print(f'Repository root: {ROOT}')
print(f'PyTorch: {torch.__version__}')

Repository root: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development
PyTorch: 2.8.0+cu126


## Conversion settings

Change `SOURCE_CHECKPOINT` when converting another run. The default points to the latest completed GO1 PACT-Pos run that contains `grf_decoder_state_dict`.

In [2]:
SOURCE_CHECKPOINT = (
    ROOT / 'logs' / 'pact_corl' / 'go1_pact_pos_rough'
    / 'Sep04_19-09-01_pact_posboot_100hz_grf' / 'model_5000.pt'
)
OUTPUT_DIR = (
    MODULES_DIR / 'pretained_checkpoints' / 'rl_pos' / 'pact_corl'
    / 'go1_pact_pos_rough' / SOURCE_CHECKPOINT.parent.name
)
OUTPUT_NAME = f'{SOURCE_CHECKPOINT.stem}_converted.pt'
OUTPUT_CHECKPOINT = OUTPUT_DIR / OUTPUT_NAME
OVERWRITE = False
INITIAL_TORQUE_STD = 1.0

assert SOURCE_CHECKPOINT.is_file(), f'Missing source checkpoint: {SOURCE_CHECKPOINT}'
print(f'Source: {SOURCE_CHECKPOINT}')
print(f'Output: {OUTPUT_CHECKPOINT}')

Source: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/logs/pact_corl/go1_pact_pos_rough/Sep04_19-09-01_pact_posboot_100hz_grf/model_5000.pt
Output: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/rsl_rl/modules/pretained_checkpoints/rl_pos/pact_corl/go1_pact_pos_rough/Sep04_19-09-01_pact_posboot_100hz_grf/model_5000_converted.pt


## Build the source and target architectures

These dimensions mirror `GO1PACTPosCfgPPO` and `GO1PACTCfgPPO` on `master`. Keeping them visible makes architecture drift fail loudly during strict checkpoint loading.

In [3]:
NUM_OBS = 57
NUM_PRIVILEGED_OBS = 57 + (50 + 38) + 143
NUM_PRIV_STACK = 5
NUM_CRITIC_OBS = NUM_PRIVILEGED_OBS * NUM_PRIV_STACK
NUM_ACTIONS = 12
NUM_OBS_HIST = 20
LATENT_DIM = 16
EXPLICIT_DIM = 3 + 4 + 4 + 1 + 1 + 3
CONTEXT_DIM = LATENT_DIM + EXPLICIT_DIM
ACTOR_LAYERS = [512, 256, 128]
CRITIC_LAYERS = [1024, 256, 128]
ENCODER_LAYERS = [256, 128]
DECODER_LAYERS = [128, 256, 512]
PRIVILEGED_DECODE_DIM = NUM_PRIVILEGED_OBS - 12
GRF_DECODER_INPUT_DIM = CONTEXT_DIM + NUM_ACTIONS

actor_kwargs = dict(
    num_actor_obs=NUM_OBS,
    num_critic_obs=NUM_CRITIC_OBS,
    num_actions=NUM_ACTIONS,
    actor_layers=ACTOR_LAYERS,
    critic_layers=CRITIC_LAYERS,
    cenet_in_dim=NUM_OBS * NUM_OBS_HIST,
    cenet_latent_dim=LATENT_DIM,
    cenet_velo_dim=EXPLICIT_DIM,
    cenet_enc_layers=ENCODER_LAYERS,
    activation='elu',
    init_noise_std=1.0,
)

source_actor = ActorCritic_PACT_Pos(**actor_kwargs)
target_actor = ActorCritic_PACT(**actor_kwargs)
source_decoder = ContextDecoder(CONTEXT_DIM, DECODER_LAYERS, PRIVILEGED_DECODE_DIM)
target_decoder = ContextDecoder(CONTEXT_DIM, DECODER_LAYERS, PRIVILEGED_DECODE_DIM)
source_grf_decoder = ContextDecoder(GRF_DECODER_INPUT_DIM, DECODER_LAYERS, 12)
target_grf_decoder = ContextDecoder(GRF_DECODER_INPUT_DIM, DECODER_LAYERS, 12)

## Load and strictly validate the PACT-Pos checkpoint

In [4]:
def without_compile_prefix(state_dict):
    return {key.removeprefix('_orig_mod.'): value for key, value in state_dict.items()}


source_checkpoint = torch.load(SOURCE_CHECKPOINT, map_location='cpu', weights_only=False)
required_keys = {
    'model_state_dict',
    'decoder_state_dict',
    'grf_decoder_state_dict',
}
missing = required_keys.difference(source_checkpoint)
if missing:
    raise KeyError(f'Source checkpoint is missing required payloads: {sorted(missing)}')

source_actor.load_state_dict(without_compile_prefix(source_checkpoint['model_state_dict']), strict=True)
source_decoder.load_state_dict(without_compile_prefix(source_checkpoint['decoder_state_dict']), strict=True)
source_grf_decoder.load_state_dict(
    without_compile_prefix(source_checkpoint['grf_decoder_state_dict']), strict=True
)
print(f"Strict source validation passed (iteration={source_checkpoint.get('iter', 'unknown')}).")

Strict source validation passed (iteration=5000).


## Transfer PACT-Pos weights into PACT

The actor is copied by named submodule because its `std` parameter changes from 12 values in PACT-Pos to 24 values in PACT.

In [5]:
TRANSFERRED_ACTOR_MODULES = (
    'context_encoder',
    'act_trunk',
    'critic',
    'act_pos_out',
    'act_tau_out',
)

for module_name in TRANSFERRED_ACTOR_MODULES:
    source_module = getattr(source_actor, module_name)
    target_module = getattr(target_actor, module_name)
    target_module.load_state_dict(source_module.state_dict(), strict=True)

with torch.no_grad():
    target_actor.std[:NUM_ACTIONS].copy_(source_actor.std)
    target_actor.std[NUM_ACTIONS:].fill_(INITIAL_TORQUE_STD)

target_decoder.load_state_dict(source_decoder.state_dict(), strict=True)
target_grf_decoder.load_state_dict(source_grf_decoder.state_dict(), strict=True)
print('Transferred:', ', '.join(TRANSFERRED_ACTOR_MODULES))
print('Transferred: privileged decoder, GRF decoder')
print(f'PACT std: position={target_actor.std[:12].tolist()}, torque={target_actor.std[12:].tolist()}')

Transferred: context_encoder, act_trunk, critic, act_pos_out, act_tau_out
Transferred: privileged decoder, GRF decoder
PACT std: position=[0.40602967143058777, 0.48257774114608765, 0.4318302869796753, 0.4032435119152069, 0.4786122441291809, 0.4330264627933502, 0.39846664667129517, 0.47062814235687256, 0.45049044489860535, 0.39981818199157715, 0.4613950550556183, 0.439096599817276], torque=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


## Validate functional equivalence before saving

In [6]:
source_actor.eval()
target_actor.eval()
source_decoder.eval()
target_decoder.eval()
source_grf_decoder.eval()
target_grf_decoder.eval()

with torch.inference_mode():
    obs = torch.randn(8, NUM_OBS)
    history = torch.randn(8, NUM_OBS * NUM_OBS_HIST)
    latent, explicit = source_actor.cenet_enc_inference(history)
    source_features = source_actor.act_trunk(torch.cat((obs, latent, explicit), dim=-1))
    target_latent, target_explicit = target_actor.cenet_enc_inference(history)
    target_features = target_actor.act_trunk(
        torch.cat((obs, target_latent, target_explicit), dim=-1)
    )
    torch.testing.assert_close(target_latent, latent, rtol=0, atol=0)
    torch.testing.assert_close(target_explicit, explicit, rtol=0, atol=0)
    torch.testing.assert_close(target_features, source_features, rtol=0, atol=0)
    torch.testing.assert_close(
        target_actor.act_pos_out(target_features),
        source_actor.act_pos_out(source_features), rtol=0, atol=0,
    )
    torch.testing.assert_close(
        target_actor.act_tau_out(target_features),
        source_actor.act_tau_out(source_features), rtol=0, atol=0,
    )
    context = torch.cat((latent, explicit), dim=-1)
    nominal_torque = torch.randn(8, NUM_ACTIONS)
    grf_input = torch.cat((context, nominal_torque), dim=-1)
    torch.testing.assert_close(
        target_decoder(context), source_decoder(context), rtol=0, atol=0,
    )
    torch.testing.assert_close(
        target_grf_decoder(grf_input), source_grf_decoder(grf_input), rtol=0, atol=0,
    )

print('Functional transfer validation passed.')

Functional transfer validation passed.


## Save and reload the hot-start checkpoint

In [7]:
if OUTPUT_CHECKPOINT.exists() and not OVERWRITE:
    raise FileExistsError(
        f'{OUTPUT_CHECKPOINT} already exists. Set OVERWRITE=True to replace it.'
    )

hot_start_checkpoint = {
    'model_state_dict': target_actor.state_dict(),
    'decoder_state_dict': target_decoder.state_dict(),
    'grf_decoder_state_dict': target_grf_decoder.state_dict(),
    'iter': 0,
    'infos': {
        'conversion': 'go1_pact_pos_to_pact_hot_start',
        'source_checkpoint': str(SOURCE_CHECKPOINT),
        'source_iteration': source_checkpoint.get('iter'),
        'initial_torque_std': INITIAL_TORQUE_STD,
        'weights_only': True,
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(hot_start_checkpoint, OUTPUT_CHECKPOINT)

reloaded = torch.load(OUTPUT_CHECKPOINT, map_location='cpu', weights_only=False)
assert set(required_keys).issubset(reloaded)
ActorCritic_PACT(**actor_kwargs).load_state_dict(reloaded['model_state_dict'], strict=True)
ContextDecoder(CONTEXT_DIM, DECODER_LAYERS, PRIVILEGED_DECODE_DIM).load_state_dict(
    reloaded['decoder_state_dict'], strict=True
)
ContextDecoder(GRF_DECODER_INPUT_DIM, DECODER_LAYERS, 12).load_state_dict(
    reloaded['grf_decoder_state_dict'], strict=True
)

print(f'Saved and strictly reloaded: {OUTPUT_CHECKPOINT}')
print(f'Size: {OUTPUT_CHECKPOINT.stat().st_size / (1024 ** 2):.2f} MiB')

Saved and strictly reloaded: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/rsl_rl/modules/pretained_checkpoints/rl_pos/pact_corl/go1_pact_pos_rough/Sep04_19-09-01_pact_posboot_100hz_grf/model_5000_converted.pt
Size: 10.80 MiB


## Configure GO1 PACT

Set `GO1PACTCfgPPO.policy.pretrained_path` to the generated checkpoint. With the default settings above, the repository-relative value is:

```python
pretrained_path = "../../rsl_rl/modules/pretained_checkpoints/rl_pos/pact_corl/go1_pact_pos_rough/Sep04_19-09-01_pact_posboot_100hz_grf/model_5000_converted.pt"
```

Use this through `pretrained_path`; do not use it with `--resume`, because optimizer states are deliberately not included.